In [ ]:
# Launcher: clone the repo if missing, otherwise pull, then compile, run, display.
import glob, os, shutil, subprocess
from IPython.display import display
from PIL import Image

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]  # -fmad=false: CPU and GPU match bit for bit
PROGRAMS = ["project", "bench", "pano"]  # src/<name>.cu -> build/<name>; pano reads project's scene.bin

assert "<" not in REPO_URL, "Set REPO_URL at the top of this cell first."
repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
build = os.path.join(repo, "build")


def sh(*cmd, cwd=None):
    """Run a command, print its output, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout, flush=True)
    if r.returncode != 0:
        raise SystemExit(f"FAILED (exit {r.returncode}): {' '.join(cmd)}")


# 1. Get the code.
if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)  # what is being run
sh("nvidia-smi", "--query-gpu=name", "--format=csv,noheader")

# 2. Compile and run each program from a fresh build/ (outputs land there, untracked).
shutil.rmtree(build, ignore_errors=True)
os.makedirs(build)
for name in PROGRAMS:
    sh("nvcc", *FLAGS, os.path.join(repo, "src", name + ".cu"), "-o", os.path.join(build, name))
    sh("./" + name, cwd=build)

# 3. Display any PGM/PPM images the programs wrote.
for path in sorted(glob.glob(os.path.join(build, "*.p[gp]m"))):
    print(os.path.basename(path))
    display(Image.open(path))